In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv('online_retail_sales_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [12]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [13]:
invoice_summary = df.groupby('Invoice').agg({
    'Quantity': 'sum',
    'TotalPrice': 'sum',
    'StockCode': 'nunique',
    'Customer ID': 'first'
}).reset_index()

In [14]:
invoice_summary.columns = ['Invoice', 'TotalQuantity', 'TotalAmount', 'UniqueProducts', 'Customer ID']

In [15]:
invoice_summary.head()

,Invoice,TotalQuantity,TotalAmount,UniqueProducts,Customer ID
0,489434,166,505.30,8,13085
1,489435,60,145.80,4,13085
2,489436,193,630.33,19,13078
3,489437,145,310.75,23,15362
4,489438,826,2286.24,17,18102


In [16]:
from sklearn.ensemble import IsolationForest

In [17]:
features = invoice_summary[['TotalQuantity', 'TotalAmount', 'UniqueProducts']]

In [18]:
iso_forest = IsolationForest(contamination=0.02, random_state=42)
invoice_summary['Anomaly'] = iso_forest.fit_predict(features)

In [19]:
invoice_summary['Anomaly'].value_counts()

,count
Anomaly,
1,36229
-1,740


In [20]:
anomalies = invoice_summary[invoice_summary['Anomaly'] == -1]
anomalies.sort_values('TotalAmount', ascending=False).head(10)

,Invoice,TotalQuantity,TotalAmount,UniqueProducts,Customer ID,Anomaly
36936,581483,80995,168469.60,1,16446,-1
20346,541431,74215,77183.60,1,12346,-1
1604,493819,25018,44051.60,94,14156,-1
26362,556444,60,38970.00,1,15098,-1
13428,524181,8172,33167.80,13,17450,-1
19001,537659,7378,31770.98,9,18102,-1
30854,567423,12572,31698.16,12,17450,-1
14625,526934,5079,26007.08,15,18102,-1
10023,515944,4992,22863.36,17,18102,-1
26548,556917,15049,22775.93,138,12415,-1


In [21]:
invoice_summary.shape

(36969, 6)

In [22]:
invoice_summary.to_csv('anomaly_results.csv', index=False)